# Station attack (summary statistics)

The basic attack gives everyone in the same group (sex, age group, channel) the same guess.
Here we add one more public source: the **official results per polling station**. Group C removed ZIP from the release, but the register still tells us each respondent's ZIP
Idea: start from the group's Red share, then push it up or down depending on whether the person's station voted more or less Red than average.

In [4]:
import pandas as pd
from datetime import date

CUTOFF = 0.80

In [5]:
released = pd.read_csv("../dataC/anonymised_dataC.csv", dtype=str)
register = pd.read_excel("../dataC/public_data_registerC.xlsx")
results  = pd.read_excel("../dataC/public_data_resultsC.xlsx")
names = [line.strip() for line in open("../dataC/survey_listC.txt", encoding="utf-8") if line.strip()]
results

,Unnamed: 0,Red,Green,Invalid ballots,Total
0,Polling station: ZIP 2100,24,100,1,125
1,Polling station: ZIP 2200,52,126,1,179
2,Polling station: ZIP 2300,100,96,6,202
3,Polling station: ZIP 2400,86,175,2,263
4,E-votes,107,182,11,300
5,Total,369,679,21,1069


In [ ]:
#build each respondent's profile
resp = register[register["name"].isin(names)].copy() #keep only the survey respondents

ref = date(2025, 7, 1)
age = resp["dob"].apply(lambda d: ref.year - d.year - ((ref.month, ref.day) < (d.month, d.day)))
resp["age_group"] = pd.cut(age, [0, 40, 60, 200], right=False, labels=["20-39", "40-59", "60+"]).astype(str)
resp["evote"] = resp["last_voted"].astype(str)
resp["station"] = resp["zip"].astype(str).where(resp["evote"] == "0", "E-votes")
profile = ["sex", "age_group", "evote"] #to be used as a key when merging with the red voters per demographic group per station
resp[["name"] + profile + ["station"]].head(100)

,name,sex,age_group,evote,station
3,"Cole, Amos",Male,20-39,1,E-votes
4,"Gibson, Isaiah",Male,20-39,1,E-votes
8,"Piester, Emma",Female,20-39,1,E-votes
24,"el-Fahmy, Fikri",Male,20-39,1,E-votes
27,"Spoon, Kelsey",Female,20-39,1,E-votes
...,...,...,...,...,...
739,"Valdez, Ismael",Male,20-39,0,2100
752,"Infante, Cristina",Female,20-39,0,2300
755,"Carter, Matthew",Male,20-39,0,2100
768,"Wanless, Brandon",Male,40-59,0,2300


get the red share for each group from the released data

In [17]:
valid = released[released["party"] != "Invalid vote"]
group_red = valid.groupby(profile)["party"].apply(lambda s: (s == "Red").mean()).rename("group_red").reset_index()
group_red

,sex,age_group,evote,group_red
0,Female,20-39,0,0.294118
1,Female,20-39,1,0.222222
2,Female,40-59,0,0.411765
3,Female,40-59,1,0.285714
4,Female,60+,0,0.684211
5,Female,60+,1,0.750000
6,Male,20-39,0,0.052632
7,Male,20-39,1,0.166667
8,Male,40-59,0,0.300000
9,Male,40-59,1,0.428571


get the red share per station (from the official results)

Then we compare each station with the average of its channel: paper stations against all paper votes, online against itself (so online voters don't move).

In [18]:
results["station"] = results.iloc[:, 0].astype(str).str.replace("Polling station: ZIP ", "") #"Polling station: ZIP 2100" becomes "2100"
results["station_red"] = results["Red"] / (results["Red"] + results["Green"]) #get the red share

paper = results[results["station"].str.isdigit()] #exclude E-votes and Total
paper_avg = paper["Red"].sum() / (paper["Red"].sum() + paper["Green"].sum()) #pooled Red share across all paper stations
results["channel_avg"] = results["station_red"].where(results["station"] == "E-votes", paper_avg) #the baseline each station is compared against #Paper stations get paper_avg, and E-votes gets its own station_red
stations = results.loc[results["station"] != "Total", ["station", "station_red", "channel_avg"]] 
stations

,station,station_red,channel_avg
0,2100,0.193548,0.345191
1,2200,0.292135,0.345191
2,2300,0.510204,0.345191
3,2400,0.329502,0.345191
4,E-votes,0.370242,0.370242


## group share, adjusted by station
Working in odds (p / (1 - p)) - new odds = starting odds × how much the new evidence favours Red. If a station votes twice as Red as its channel average (in odds), we double the person's Red odds:

`odds(Red) = odds(group) × odds(station) / odds(channel average)`

In [19]:
odds = lambda p: p / (1 - p)

resp = resp.merge(group_red, on=profile).merge(stations, on="station")
o = odds(resp["group_red"]) * odds(resp["station_red"]) / odds(resp["channel_avg"])
resp["p_red"] = o / (1 + o) #convert odds back to probability

resp["guess"] = resp["p_red"].apply(lambda p: "Red" if p >= 0.5 else "Green")
resp["confidence"] = resp["p_red"].where(resp["guess"] == "Red", 1 - resp["p_red"])
print("expected accuracy over all 200:", round(resp["confidence"].mean(), 3))
resp[["name"] + profile + ["station", "group_red", "p_red", "guess", "confidence"]].sort_values("confidence", ascending=False).head(10)

expected accuracy over all 200: 0.696


,name,sex,age_group,evote,station,group_red,p_red,guess,confidence
55,"el-Kazmi, Nizaam",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
64,"Cordiner, Darren",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
61,"Bittenbinder, Paul",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
63,"Wang, Vinh",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
95,"Valdez, Ismael",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
97,"Carter, Matthew",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
78,"White, Ronald",Male,20-39,0,2100,0.052632,0.024669,Green,0.975331
71,"Cheatham, Tevin",Male,20-39,0,2200,0.052632,0.041680,Green,0.958320
81,"al-Adel, Mahuroos",Male,20-39,0,2400,0.052632,0.049240,Green,0.950760
83,"Leidig, Jeffrey",Male,20-39,0,2400,0.052632,0.049240,Green,0.950760


## Compare with the basic attack and write the confident guesses

In [10]:
resp["basic_conf"] = resp["group_red"].where(resp["group_red"] >= 0.5, 1 - resp["group_red"])
print("confidence >= CUTOFF, basic  :", int((resp["basic_conf"] >= CUTOFF).sum()))
print("confidence >= CUTOFF, station:", int((resp["confidence"] >= CUTOFF).sum()))

sure = resp[resp["confidence"] >= CUTOFF].sort_values("confidence", ascending=False)
with open("station_deanonymised_C.csv", "w", encoding="utf-8", newline="") as f:
    for name, party in zip(sure["name"], sure["guess"]):
        f.write(f"{name}; {party.lower()}\n")

print(f"{len(sure)} voters written, expected correct {sure['confidence'].sum():.1f}")
sure.groupby(profile + ["station", "guess"]).size()

confidence >= CUTOFF, basic  : 32
confidence >= CUTOFF, station: 45
45 voters written, expected correct 39.5


sex     age_group  evote  station  guess
Female  20-39      0      2100     Green     6
        60+        0      2300     Red       5
Male    20-39      0      2100     Green     7
                          2200     Green     1
                          2300     Green    10
                          2400     Green     2
                   1      E-votes  Green    12
        40-59      0      2100     Green     2
dtype: int64